In [1]:
# setup

from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Proyek_CRM_KELOMPOK')
sys.path.append(str(PROJECT_DIR))

from project_config import CLEAN_DATA_PATH, PROCESSED_DIR, OUTPUT_DIR, REPORT_DIR

import pandas as pd

from sklearn.model_selection import train_test_split

Mounted at /content/drive


In [2]:
#load data clean
df = pd.read_csv(CLEAN_DATA_PATH)

print(df.shape)
df.head()

(286, 56)


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,...,High Quality of package,Number of calls,Politeness,Freshness,Temperature,Good Taste,Good Quantity,Output,Reviews,churn_risk
0,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,...,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Yes,Nil,0
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,...,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,Yes,Nil,0
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,...,Very Important,Moderately Important,Very Important,Very Important,Important,Very Important,Moderately Important,Yes,"Many a times payment gateways are an issue, so...",0
3,22,Female,Single,Student,No Income,Graduate,6,12.9473,77.5616,560019,...,Important,Moderately Important,Very Important,Very Important,Very Important,Very Important,Important,Yes,nil,0
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,12.9850,77.5533,560010,...,Important,Moderately Important,Important,Important,Important,Very Important,Very Important,Yes,NIL,0


In [3]:
# menentukan drop kolom reviews
drop_for_modeling = ['Output']

if 'Reviews' in df.columns:
    drop_for_modeling.append('Reviews')

modeling_df = df.drop(columns=drop_for_modeling)

modeling_df.head()

,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,...,Influence of rating,Less Delivery time,High Quality of package,Number of calls,Politeness,Freshness,Temperature,Good Taste,Good Quantity,churn_risk
0,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,...,Yes,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,0
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,...,Yes,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,0
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,...,Yes,Important,Very Important,Moderately Important,Very Important,Very Important,Important,Very Important,Moderately Important,0
3,22,Female,Single,Student,No Income,Graduate,6,12.9473,77.5616,560019,...,Yes,Very Important,Important,Moderately Important,Very Important,Very Important,Very Important,Very Important,Important,0
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,12.9850,77.5533,560010,...,Yes,Important,Important,Moderately Important,Important,Important,Important,Very Important,Very Important,0


In [4]:
#simpan dataset modelling
modeling_data_path = PROCESSED_DIR / 'modeling_dataset.csv'
modeling_df.to_csv(modeling_data_path, index=False)

print('Modeling dataset disimpan:', modeling_data_path)

Modeling dataset disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/data/processed/modeling_dataset.csv


In [5]:
# Pisahkan fitur dan target
X = modeling_df.drop(columns=['churn_risk'])
y = modeling_df['churn_risk']

In [6]:
#metadata fitur
feature_metadata = pd.DataFrame({
    'feature_name': X.columns,
    'data_type': X.dtypes.astype(str).values,
    'unique_count': X.nunique().values,
    'missing_count': X.isnull().sum().values
})

feature_metadata['feature_group'] = feature_metadata['data_type'].apply(
    lambda x: 'numeric' if x in ['int64', 'float64'] else 'categorical'
)

feature_metadata

,feature_name,data_type,unique_count,missing_count,feature_group
0,Age,int64,16,0,numeric
1,Gender,object,2,0,categorical
2,Marital Status,object,3,0,categorical
3,Occupation,object,4,0,categorical
4,Monthly Income,object,5,0,categorical
5,Educational Qualifications,object,5,0,categorical
6,Family size,int64,6,0,numeric
7,latitude,float64,77,0,numeric
8,longitude,float64,76,0,numeric
9,Pin code,int64,77,0,numeric


In [7]:
# Simpan metadata fitur
feature_metadata_path = OUTPUT_DIR / 'feature_metadata.csv'
feature_metadata.to_csv(feature_metadata_path, index=False)

print('Feature metadata disimpan:', feature_metadata_path)

Feature metadata disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/feature_metadata.csv


In [8]:
# split train tes
train_index, test_index = train_test_split(
    modeling_df.index,
    test_size=0.2,
    stratify=y,
    random_state=42
)

split_index = pd.DataFrame({
    'index': modeling_df.index,
    'split': ['train' if i in train_index else 'test' for i in modeling_df.index]
})

split_index.head()

,index,split
0,0,train
1,1,test
2,2,train
3,3,train
4,4,train


In [9]:
# simpan index train text
split_path = OUTPUT_DIR / 'train_test_index.csv'
split_index.to_csv(split_path, index=False)

print('Train test index disimpan:', split_path)

Train test index disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/train_test_index.csv


In [10]:
# simpan catatan preprocessing
notes = """
Catatan Preprocessing

1. Target churn_risk dibuat dari Output.
2. Output dihapus dari fitur karena sudah menjadi sumber target.
3. Reviews tidak dipakai dalam model utama karena berbentuk teks.
4. Fitur numerik akan diproses dengan median imputer dan standard scaler.
5. Fitur kategorikal akan diproses dengan most frequent imputer dan one hot encoding.
6. Split data memakai stratify agar proporsi kelas tetap seimbang.
"""

notes_path = REPORT_DIR / 'preprocessing_notes.txt'

with open(notes_path, 'w') as f:
    f.write(notes)

print('Preprocessing notes disimpan:', notes_path)

Preprocessing notes disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/reports/preprocessing_notes.txt
